# 03 · Compare all completed models

Run after at least two model days are complete. Only compatible controlled
2-class runs are accepted. Missing or failed models remain visibly missing;
the notebook never creates placeholder metrics.


In [ ]:
BENCHMARK_TRACK = "controlled"
COMPARISON_VERSION = "v2"

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git"
REPOSITORY_BRANCH = "main"
SMOKE_TEST = os.environ.get("SMOKE_TEST", "").lower() in {"1", "true", "yes"}

# The only logic a notebook still owns: make `src` importable. Everything after
# this line - Git state, platform detection, paths, dependency policy - lives in
# src/notebook_bootstrap.py so all notebooks behave identically.
_override = os.environ.get("BENCHMARK_REPO_ROOT")
_candidates = (
    [Path(_override).expanduser()]
    if _override
    else [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/content/aerial-object-detection-benchmark"),
        Path("/kaggle/working/aerial-object-detection-benchmark"),
    ]
)
REPO_PATH = next(
    (
        candidate.resolve()
        for candidate in _candidates
        if (candidate / "src" / "notebook_bootstrap.py").is_file()
    ),
    None,
)
if REPO_PATH is None:
    _host = (
        Path("/content")
        if Path("/content").is_dir()
        else Path("/kaggle/working")
        if Path("/kaggle/working").is_dir()
        else None
    )
    if _host is None:
        raise RuntimeError(
            "Run this notebook from the repository, or set BENCHMARK_REPO_ROOT "
            "to an existing clone."
        )
    REPO_PATH = (_host / "aerial-object-detection-benchmark").resolve()
    REPO_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_PATH)],
        check=True,
    )
sys.path.insert(0, str(REPO_PATH))

from src.notebook_bootstrap import bootstrap_notebook

bootstrap = bootstrap_notebook(
    REPO_PATH,
    requirements_file=None,
    use_google_drive=True,
    smoke_test=SMOKE_TEST,
)
notebook_environment = bootstrap.environment
REPO_PATH = notebook_environment.repository_root
DRIVE_ROOT = notebook_environment.artifact_root
LOCAL_CACHE_ROOT = notebook_environment.local_cache_root
NOTEBOOK_PLATFORM = notebook_environment.platform
IN_COLAB = NOTEBOOK_PLATFORM == "colab"
IN_KAGGLE = NOTEBOOK_PLATFORM == "kaggle"
print(bootstrap.summary())


In [ ]:
if SMOKE_TEST:
    result = {"message": "Smoke mode: comparison requires at least two measured completed models."}
else:
    from src.config.benchmark_tracks import load_track_config
    from src.workflows.comparison import compare_completed_models
    from src.workflows.versioned_comparison import build_comparison_tables
    track_config = load_track_config(REPO_PATH, BENCHMARK_TRACK)
    result = compare_completed_models(DRIVE_ROOT, benchmark_track=BENCHMARK_TRACK)
print(json.dumps(result, indent=2, default=str))


In [ ]:
if not SMOKE_TEST:
    from IPython.display import Image, Markdown, display
    report = Path(result["output"]) / "comparison.md"
    display(Markdown(report.read_text(encoding="utf-8")))
    figure = Path(result["output"]) / "accuracy_latency.png"
    if figure.is_file():
        display(Image(filename=str(figure)))
